# 🏐 Lab 1 — Web Crawling + NER
**Course:** Web Mining & Semantics  
**Domain:** International Men's Volleyball (FIVB)  
**Sources:** Wikipedia + Wikidata  

### Outputs
- `crawler_output.jsonl` — cleaned text + metadata per URL
- `extracted_knowledge.csv` — entities (type, text, source URL)

## ⚙️ 0. Setup & Installations

In [ ]:
# Install dependencies
!pip install -q trafilatura httpx SPARQLWrapper pandas
!pip install -q spacy
!python -m spacy download en_core_web_trf -q

In [ ]:
# Mount Google Drive to save outputs
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/volleyball-kg/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Imports
import json
import time
import httpx
import trafilatura
import pandas as pd
import spacy
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.parse import urljoin, urlparse
from collections import defaultdict

# Load spaCy model
nlp = spacy.load('en_core_web_trf')
print('All imports OK ✅')

## 🌐 1. Phase 1 — Wikipedia Crawling

In [ ]:
# ── Seed URLs ──────────────────────────────────────────────────────────────
# Covers: national teams, players, tournaments, clubs
SEED_URLS = [
    # National Teams
    'https://en.wikipedia.org/wiki/Brazil_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/France_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Poland_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Italy_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/United_States_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Japan_men%27s_national_volleyball_team',
    'https://en.wikipedia.org/wiki/Argentina_men%27s_national_volleyball_team',
    # Tournaments
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_Nations_League',
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_World_Championship',
    'https://en.wikipedia.org/wiki/Volleyball_at_the_Summer_Olympics',
    'https://en.wikipedia.org/wiki/FIVB_Volleyball_World_Cup',
    # Players
    'https://en.wikipedia.org/wiki/Earvin_Ngapeth',
    'https://en.wikipedia.org/wiki/Wilfredo_Le%C3%B3n',
    'https://en.wikipedia.org/wiki/Nimir_Abdel-Aziz',
    'https://en.wikipedia.org/wiki/Giba',
    'https://en.wikipedia.org/wiki/Bartosz_Kurek',
    'https://en.wikipedia.org/wiki/Ivan_Zaytsev',
    'https://en.wikipedia.org/wiki/Matthew_Anderson_(volleyball)',
    'https://en.wikipedia.org/wiki/Bruno_Rezende',
    # Clubs
    'https://en.wikipedia.org/wiki/Trentino_Volley',
    'https://en.wikipedia.org/wiki/Sir_Sicoma_Monini_Perugia',
    'https://en.wikipedia.org/wiki/Paris_Volley',
    # Federations
    'https://en.wikipedia.org/wiki/FIVB',
    'https://en.wikipedia.org/wiki/Confédération_Européenne_de_Volleyball',
]

print(f'Total seed URLs: {len(SEED_URLS)}')

In [ ]:
def is_useful_page(text: str, min_words: int = 500) -> bool:
    """Check if extracted text is long enough to be useful."""
    return len(text.split()) >= min_words


def fetch_and_clean(url: str) -> dict | None:
    """
    Fetch a URL and extract its main content using trafilatura.
    Returns a dict with url, title, text, word_count — or None if not useful.
    """
    try:
        # Fetch raw HTML
        response = httpx.get(url, timeout=15, follow_redirects=True,
                             headers={'User-Agent': 'VolleyballKG-Bot/1.0 (academic project)'})
        response.raise_for_status()

        # Extract main content (removes boilerplate, ads, nav)
        text = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=True,
            no_fallback=False
        )

        if not text:
            print(f'  [SKIP] No content extracted: {url}')
            return None

        if not is_useful_page(text):
            print(f'  [SKIP] Too short ({len(text.split())} words): {url}')
            return None

        # Extract title from URL slug
        title = urlparse(url).path.split('/')[-1].replace('_', ' ')

        return {
            'url': url,
            'title': title,
            'text': text,
            'word_count': len(text.split())
        }

    except Exception as e:
        print(f'  [ERROR] {url}: {e}')
        return None

In [ ]:
# ── Crawl all seed URLs ────────────────────────────────────────────────────
crawled_pages = []
DELAY_SECONDS = 1.5  # Be polite — respect robots.txt spirit

print('Starting Wikipedia crawl...\n')
for i, url in enumerate(SEED_URLS):
    print(f'[{i+1}/{len(SEED_URLS)}] Fetching: {url}')
    result = fetch_and_clean(url)
    if result:
        crawled_pages.append(result)
        print(f'  [OK] {result["word_count"]} words')
    time.sleep(DELAY_SECONDS)

print(f'\n✅ Successfully crawled: {len(crawled_pages)}/{len(SEED_URLS)} pages')

In [ ]:
# ── Save to JSONL ──────────────────────────────────────────────────────────
JSONL_PATH = f'{OUTPUT_DIR}/crawler_output.jsonl'

with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for page in crawled_pages:
        f.write(json.dumps(page, ensure_ascii=False) + '\n')

print(f'Saved {len(crawled_pages)} pages → {JSONL_PATH}')

## 🗺️ 2. Phase 1b — Wikidata SPARQL Fetcher

In [ ]:
def query_wikidata(sparql_query: str) -> list[dict]:
    """Execute a SPARQL query on Wikidata and return results as a list of dicts."""
    endpoint = SPARQLWrapper('https://query.wikidata.org/sparql')
    endpoint.setQuery(sparql_query)
    endpoint.setReturnFormat(JSON)
    endpoint.addCustomHttpHeader('User-Agent', 'VolleyballKG-Bot/1.0 (academic project)')
    results = endpoint.query().convert()
    return results['results']['bindings']

In [ ]:
# ── Query 1: Male volleyball players ──────────────────────────────────────
QUERY_PLAYERS = """
SELECT DISTINCT ?player ?playerLabel ?nationalityLabel ?birthDate ?positionLabel WHERE {
  ?player wdt:P31 wd:Q5 ;                      # is a human
          wdt:P106 wd:Q15117302 ;               # occupation: volleyball player
          wdt:P21 wd:Q6581097 .                 # gender: male
  OPTIONAL { ?player wdt:P27 ?nationality . }
  OPTIONAL { ?player wdt:P569 ?birthDate . }
  OPTIONAL { ?player wdt:P413 ?position . }     # playing position
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 500
"""

print('Querying Wikidata: volleyball players...')
players_raw = query_wikidata(QUERY_PLAYERS)
print(f'Found {len(players_raw)} players')

In [ ]:
# ── Query 2: Men's national volleyball teams ───────────────────────────────
QUERY_TEAMS = """
SELECT DISTINCT ?team ?teamLabel ?countryLabel ?federationLabel WHERE {
  ?team wdt:P31 wd:Q6979593 .                   # men's national volleyball team
  OPTIONAL { ?team wdt:P17 ?country . }
  OPTIONAL { ?team wdt:P118 ?federation . }      # league/federation
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 200
"""

print('Querying Wikidata: national teams...')
teams_raw = query_wikidata(QUERY_TEAMS)
print(f'Found {len(teams_raw)} teams')

In [ ]:
# ── Query 3: Volleyball tournaments/competitions ───────────────────────────
QUERY_TOURNAMENTS = """
SELECT DISTINCT ?tournament ?tournamentLabel ?organizerLabel ?inceptionYear WHERE {
  ?tournament wdt:P31/wdt:P279* wd:Q13406554 .  # volleyball competition
  OPTIONAL { ?tournament wdt:P664 ?organizer . }
  OPTIONAL { ?tournament wdt:P571 ?inception . 
             BIND(YEAR(?inception) AS ?inceptionYear) }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 100
"""

print('Querying Wikidata: tournaments...')
tournaments_raw = query_wikidata(QUERY_TOURNAMENTS)
print(f'Found {len(tournaments_raw)} tournaments')

In [ ]:
# ── Query 4: Player ↔ Team membership ─────────────────────────────────────
QUERY_MEMBERSHIPS = """
SELECT DISTINCT ?playerLabel ?teamLabel WHERE {
  ?player wdt:P31 wd:Q5 ;
          wdt:P106 wd:Q15117302 ;
          wdt:P21 wd:Q6581097 ;
          wdt:P54 ?team .                        # member of sports team
  ?team wdt:P31 wd:Q6979593 .                   # men's national volleyball team
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 500
"""

print('Querying Wikidata: player-team memberships...')
memberships_raw = query_wikidata(QUERY_MEMBERSHIPS)
print(f'Found {len(memberships_raw)} memberships')

In [ ]:
# ── Save Wikidata results as structured JSONL ──────────────────────────────
WIKIDATA_PATH = f'{OUTPUT_DIR}/wikidata_output.jsonl'

wikidata_results = {
    'players': players_raw,
    'teams': teams_raw,
    'tournaments': tournaments_raw,
    'memberships': memberships_raw
}

with open(WIKIDATA_PATH, 'w', encoding='utf-8') as f:
    for category, records in wikidata_results.items():
        for record in records:
            row = {'source': 'wikidata', 'category': category}
            row.update({k: v['value'] for k, v in record.items()})
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

total = sum(len(v) for v in wikidata_results.values())
print(f'Saved {total} Wikidata records → {WIKIDATA_PATH}')

## 🏷️ 3. Phase 2 — Named Entity Recognition (NER)

In [ ]:
# Entity types we care about for volleyball KG
TARGET_LABELS = {'PERSON', 'ORG', 'GPE', 'EVENT', 'DATE', 'LOC'}

def extract_entities(page: dict) -> list[dict]:
    """
    Run spaCy NER on a crawled page.
    Returns a list of entity records with type, text, sentence context, and source URL.
    """
    entities = []
    # Process in chunks to avoid memory issues with long texts
    text = page['text'][:50000]  # Cap at 50k chars for Colab memory
    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ not in TARGET_LABELS:
            continue
        # Filter out very short or very long entities (likely noise)
        if len(ent.text.strip()) < 2 or len(ent.text.split()) > 6:
            continue
        entities.append({
            'entity_text': ent.text.strip(),
            'entity_type': ent.label_,
            'source_url': page['url'],
            'source_title': page['title'],
            'sentence': ent.sent.text.strip()[:200]  # context snippet
        })

    return entities

In [ ]:
# ── Run NER on all crawled pages ───────────────────────────────────────────
all_entities = []

print('Running NER on crawled pages...\n')
for i, page in enumerate(crawled_pages):
    print(f'[{i+1}/{len(crawled_pages)}] Processing: {page["title"]}')
    entities = extract_entities(page)
    all_entities.extend(entities)
    print(f'  → {len(entities)} entities found')

print(f'\n✅ Total entities extracted: {len(all_entities)}')

## 🔗 4. Phase 2b — Relation Extraction (Dependency Parsing)

In [ ]:
def extract_relations(page: dict) -> list[dict]:
    """
    Extract candidate (subject, verb, object) triples using dependency parsing.
    Looks for sentences where two named entities are connected by a verb.
    """
    relations = []
    text = page['text'][:30000]
    doc = nlp(text)

    for sent in doc.sents:
        ents_in_sent = [e for e in sent.ents if e.label_ in TARGET_LABELS]
        if len(ents_in_sent) < 2:
            continue

        # Look for nsubj → verb → dobj/prep patterns
        for token in sent:
            if token.pos_ != 'VERB':
                continue
            subject = None
            obj = None
            for child in token.children:
                if child.dep_ == 'nsubj':
                    # Check if this child is part of a named entity
                    for ent in ents_in_sent:
                        if child.i >= ent.start and child.i < ent.end:
                            subject = ent.text
                if child.dep_ in ('dobj', 'pobj', 'attr'):
                    for ent in ents_in_sent:
                        if child.i >= ent.start and child.i < ent.end:
                            obj = ent.text

            if subject and obj and subject != obj:
                relations.append({
                    'subject': subject,
                    'relation': token.lemma_,
                    'object': obj,
                    'sentence': sent.text.strip()[:200],
                    'source_url': page['url']
                })

    return relations

In [ ]:
# ── Run relation extraction ────────────────────────────────────────────────
all_relations = []

print('Running relation extraction...\n')
for page in crawled_pages:
    rels = extract_relations(page)
    all_relations.extend(rels)

print(f'✅ Total candidate relations extracted: {len(all_relations)}')
# Preview
pd.DataFrame(all_relations).head(10)

## 💾 5. Export Final Deliverables

In [ ]:
# ── Export extracted_knowledge.csv ─────────────────────────────────────────
df_entities = pd.DataFrame(all_entities)

# Deduplicate: same entity text + type + source
df_entities = df_entities.drop_duplicates(subset=['entity_text', 'entity_type', 'source_url'])

CSV_PATH = f'{OUTPUT_DIR}/extracted_knowledge.csv'
df_entities.to_csv(CSV_PATH, index=False, encoding='utf-8')

print(f'Saved {len(df_entities)} unique entities → {CSV_PATH}')
print('\nEntity type distribution:')
print(df_entities['entity_type'].value_counts())

In [ ]:
# ── Export relations CSV ───────────────────────────────────────────────────
df_relations = pd.DataFrame(all_relations)
df_relations = df_relations.drop_duplicates(subset=['subject', 'relation', 'object'])

REL_PATH = f'{OUTPUT_DIR}/extracted_relations.csv'
df_relations.to_csv(REL_PATH, index=False, encoding='utf-8')

print(f'Saved {len(df_relations)} unique relations → {REL_PATH}')
print('\nTop 15 most frequent relation verbs:')
print(df_relations['relation'].value_counts().head(15))

## 📊 6. Summary & Ambiguity Analysis (Report Section)

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────
print('=' * 50)
print('LAB 1 — SUMMARY')
print('=' * 50)
print(f'Pages crawled (Wikipedia):  {len(crawled_pages)}')
print(f'Wikidata records fetched:   {total}')
print(f'Unique entities extracted:  {len(df_entities)}')
print(f'Candidate relations found:  {len(df_relations)}')
print()
print('Entity breakdown:')
for label, count in df_entities['entity_type'].value_counts().items():
    print(f'  {label:<8} {count}')

In [ ]:
# ── Ambiguity Analysis (required for report) ───────────────────────────────
# Find entities where same text appears with different types
ambiguity_cases = (
    df_entities.groupby('entity_text')['entity_type']
    .nunique()
    .reset_index()
    .query('entity_type > 1')
    .sort_values('entity_type', ascending=False)
)

print(f'Entities with ambiguous types: {len(ambiguity_cases)}')
print('\nTop ambiguous entities (same text, different types):')
print(ambiguity_cases.head(10))

# Show examples with their contexts
print('\n--- Ambiguity Case Examples (for report) ---')
for entity in ambiguity_cases.head(3)['entity_text']:
    cases = df_entities[df_entities['entity_text'] == entity][['entity_type', 'sentence', 'source_title']]
    print(f'\nEntity: "{entity}"')
    print(cases.to_string())

In [ ]:
print('\n✅ Lab 1 complete!')
print('Files saved to Google Drive:')
print(f'  - {OUTPUT_DIR}/crawler_output.jsonl')
print(f'  - {OUTPUT_DIR}/wikidata_output.jsonl')
print(f'  - {OUTPUT_DIR}/extracted_knowledge.csv')
print(f'  - {OUTPUT_DIR}/extracted_relations.csv')
print('\nNext step: Lab 2 — KB Construction & Alignment')